# Annotation Union Merge + Label Breakdown Visualizations

This notebook merges Reva and Ryan multi-label annotations with the requested strategy:

1. Parse labels with blank/NaN -> `_unknown`
2. Union label sets
3. Remove `_unknown` when substantive labels exist
4. Serialize output labels deterministically

It writes:
- `merged_labels.tsv`
- `merged_glossary.tsv`

and visualizes per-label breakdowns for Reva vs Ryan vs merged outputs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Resolve annotations directory whether notebook is run from annotations/ or repo root.
CWD = Path.cwd()
ANNOTATIONS_DIR = CWD if (CWD / "reva_labels.tsv").exists() else CWD / "annotations"

if not (ANNOTATIONS_DIR / "reva_labels.tsv").exists():
    raise FileNotFoundError(f"Could not find annotation files in {ANNOTATIONS_DIR}")

ANNOTATIONS_DIR

In [ ]:
def parse_labels(value) -> frozenset:
    """Parse comma-separated label string to frozenset of stripped labels.
    NaN, empty string, and whitespace-only all return frozenset({'_unknown'})."""
    if pd.isna(value) or str(value).strip() == "":
        return frozenset({'_unknown'})
    return frozenset(lbl.strip() for lbl in str(value).split(",") if lbl.strip())


def clean_merged(label_set: frozenset) -> frozenset:
    """Remove _unknown from sets that also contain substantive labels.
    Keep _unknown only when it is the sole label."""
    real = label_set - {'_unknown'}
    return real if real else frozenset({'_unknown'})


def serialize(label_set: frozenset) -> str:
    """Sort labels alphabetically and join with ', '."""
    return ", ".join(sorted(label_set))


def explode_counts(series_of_sets: pd.Series) -> pd.Series:
    """Count label frequency across items (multi-label explode)."""
    exploded = pd.Series([label for s in series_of_sets for label in s])
    return exploded.value_counts().sort_values(ascending=False)


def merge_task(
    reva_path: Path,
    ryan_path: Path,
    join_keys: list[str],
    label_col: str,
    merged_col: str,
    out_path: Path,
):
    reva = pd.read_csv(reva_path, sep="\t", dtype=str)
    ryan = pd.read_csv(ryan_path, sep="\t", dtype=str)

    merged = reva.merge(
        ryan[[*join_keys, label_col]],
        on=join_keys,
        how="inner",
        suffixes=("_reva_orig", "_ryan_orig"),
    )

    # Reconstruct the reva schema column name because merge suffixes rename overlaps.
    merged[label_col] = merged[f"{label_col}_reva_orig"]

    # Preserve original strings in explicit columns.
    merged[f"{label_col}_reva"] = merged[f"{label_col}_reva_orig"]
    merged[f"{label_col}_ryan"] = merged[f"{label_col}_ryan_orig"]

    reva_sets = merged[f"{label_col}_reva"].apply(parse_labels)
    ryan_sets = merged[f"{label_col}_ryan"].apply(parse_labels)

    merged_raw = [a | b for a, b in zip(reva_sets, ryan_sets)]
    merged_clean = [clean_merged(s) for s in merged_raw]

    merged[merged_col] = [serialize(s) for s in merged_clean]

    # Output file: all columns from reva file + required merge columns.
    base_cols = list(reva.columns)
    out_cols = [*base_cols, f"{label_col}_reva", f"{label_col}_ryan", merged_col]
    merged_out = merged[out_cols].copy()
    merged_out.to_csv(out_path, sep="\t", index=False)

    both_unknown = sum(
        (a == frozenset({'_unknown'}) and b == frozenset({'_unknown'}))
        for a, b in zip(reva_sets, ryan_sets)
    )
    unknown_stripped = sum(
        ('_unknown' in raw and cleaned != raw)
        for raw, cleaned in zip(merged_raw, merged_clean)
    )
    unique_labels = len(set().union(*merged_clean)) if merged_clean else 0

    stats = {
        'total_items': int(len(merged_out)),
        'both_unknown': int(both_unknown),
        'unknown_stripped': int(unknown_stripped),
        'unique_labels': int(unique_labels),
    }

    counts = pd.DataFrame({
        'Reva': explode_counts(reva_sets),
        'Ryan': explode_counts(ryan_sets),
        'Merged': explode_counts(pd.Series(merged_clean)),
    }).fillna(0).astype(int).sort_values('Merged', ascending=False)

    diagnostics = pd.DataFrame([
        {'category': 'both_unknown', 'count': int(both_unknown)},
        {'category': 'unknown_stripped', 'count': int(unknown_stripped)},
        {
            'category': 'unknown_retained_only',
            'count': int(sum(s == frozenset({'_unknown'}) for s in merged_clean)),
        },
    ])

    return merged_out, stats, counts, diagnostics

In [ ]:
labels_merged, labels_stats, labels_counts, labels_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_labels.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_labels.tsv",
    join_keys=["dataset", "raw_target_label"],
    label_col="dest_label",
    merged_col="dest_label_merged",
    out_path=ANNOTATIONS_DIR / "merged_labels.tsv",
)

glossary_merged, glossary_stats, glossary_counts, glossary_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_glossary.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_glossary.tsv",
    join_keys=["term"],
    label_col="inferred_target",
    merged_col="inferred_target_merged",
    out_path=ANNOTATIONS_DIR / "merged_glossary.tsv",
)

print("Wrote:")
print(ANNOTATIONS_DIR / "merged_labels.tsv")
print(ANNOTATIONS_DIR / "merged_glossary.tsv")

In [ ]:
print("Labels task")
print("-----------")
print(f"Total items:                 {labels_stats['total_items']}")
print(f"Both _unknown after merge:   {labels_stats['both_unknown']}")
print(f"_unknown stripped from union: {labels_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {labels_stats['unique_labels']}")
print()
print("Glossary task")
print("-------------")
print(f"Total items:                 {glossary_stats['total_items']}")
print(f"Both _unknown after merge:   {glossary_stats['both_unknown']}")
print(f"_unknown stripped from union: {glossary_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {glossary_stats['unique_labels']}")

In [ ]:
def plot_top_label_breakdown(counts_df: pd.DataFrame, title: str, top_n: int = 20) -> None:
    top = counts_df.head(top_n).copy()

    ax = top[["Reva", "Ryan", "Merged"]].plot(
        kind="bar",
        figsize=(14, 6),
        width=0.85,
    )
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("Count across items")
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_unknown_diagnostics(diag_df: pd.DataFrame, title: str) -> None:
    ax = diag_df.plot(kind="bar", x="category", y="count", legend=False, figsize=(8, 4), color="#5a88c8")
    ax.set_title(title)
    ax.set_xlabel("Diagnostic category")
    ax.set_ylabel("Count")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## Visual Breakdown: Labels Task

In [ ]:
plot_top_label_breakdown(labels_counts, "Labels Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(labels_diag, "Labels Task: _unknown Handling Diagnostics")
labels_counts.head(20)

## Visual Breakdown: Glossary Task

In [ ]:
plot_top_label_breakdown(glossary_counts, "Glossary Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(glossary_diag, "Glossary Task: _unknown Handling Diagnostics")
glossary_counts.head(20)